# SuperStore Sales & Profitability Analytics

Portfolio-ready retail data analytics project covering sales, profit, discounts, returns, products, customers, regions and seasonality.

**Stack:** Python, Pandas, NumPy, Matplotlib, Seaborn, Plotly, Excel, Jupyter/Google Colab.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

paths = [Path('SuperStore.xls'), Path('Datasets/SuperStore.xls'), Path('/content/SuperStore.xls')]
file_path = next((p for p in paths if p.exists()), None)
if file_path is None:
    raise FileNotFoundError('SuperStore.xls not found. Upload it to Colab or place it beside this notebook.')

orders = pd.read_excel(file_path, sheet_name='Orders')
people = pd.read_excel(file_path, sheet_name='People')
returns = pd.read_excel(file_path, sheet_name='Returns')

orders = orders.drop_duplicates().copy()
orders['Order Date'] = pd.to_datetime(orders['Order Date'], errors='coerce')
orders['Ship Date'] = pd.to_datetime(orders['Ship Date'], errors='coerce')
orders['Year'] = orders['Order Date'].dt.year
orders['Month'] = orders['Order Date'].dt.month
orders['Quarter'] = orders['Order Date'].dt.quarter
orders['Year Month'] = orders['Order Date'].dt.to_period('M').astype(str)
orders['Shipping Days'] = (orders['Ship Date'] - orders['Order Date']).dt.days
orders['Profit Margin'] = np.where(orders['Sales'] != 0, orders['Profit'] / orders['Sales'] * 100, 0)

display(orders.head())
print('Orders:', orders.shape, '| People:', people.shape, '| Returns:', returns.shape)
print('Duplicate rows removed:', orders.duplicated().sum())
display(pd.DataFrame({'Missing': orders.isna().sum(), 'Missing %': orders.isna().mean().mul(100).round(2)}).sort_values('Missing', ascending=False).head(15))


In [ ]:
# Executive KPIs
total_sales = orders['Sales'].sum()
total_profit = orders['Profit'].sum()
total_orders = orders['Order ID'].nunique()
total_customers = orders['Customer ID'].nunique()
total_quantity = orders['Quantity'].sum()
margin = total_profit / total_sales * 100
aov = total_sales / total_orders

print('=' * 65)
print('SUPERSTORE EXECUTIVE INSIGHTS')
print('=' * 65)
print(f'Total Sales        : ${total_sales:,.2f}')
print(f'Total Profit       : ${total_profit:,.2f}')
print(f'Profit Margin      : {margin:.2f}%')
print(f'Total Orders       : {total_orders:,}')
print(f'Total Customers    : {total_customers:,}')
print(f'Quantity Sold      : {total_quantity:,}')
print(f'Average Order Value: ${aov:,.2f}')


In [ ]:
# Core EDA
yearly = orders.groupby('Year', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
category = orders.groupby('Category', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
region = orders.groupby('Region', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
subcategory = orders.groupby('Sub-Category', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
top_products = orders.groupby('Product Name', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum')).sort_values('Sales', ascending=False).head(10)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sns.lineplot(data=yearly, x='Year', y='Sales', marker='o', ax=axes[0,0], label='Sales')
sns.lineplot(data=yearly, x='Year', y='Profit', marker='o', ax=axes[0,0], label='Profit')
axes[0,0].set_title('Yearly Sales & Profit')
sns.barplot(data=category, x='Category', y='Sales', ax=axes[0,1])
axes[0,1].set_title('Sales by Category')
sns.barplot(data=region, x='Region', y='Profit', ax=axes[1,0])
axes[1,0].set_title('Profit by Region')
sns.barplot(data=subcategory.sort_values('Profit'), x='Profit', y='Sub-Category', ax=axes[1,1])
axes[1,1].set_title('Sub-Category Profitability')
plt.tight_layout(); plt.show()

display(top_products)


In [ ]:
# Discount, returns, time and product analysis
orders['Discount Level'] = pd.cut(orders['Discount'], bins=[-0.01, 0.10, 0.25, 0.40, 1.0], labels=['Low','Medium','High','Very High'])
discount = orders.groupby('Discount Level', observed=True, as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
discount['Profit Margin'] = np.where(discount['Sales'] != 0, discount['Profit']/discount['Sales']*100, 0)
monthly = orders.groupby('Year Month', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
monthly['Date'] = pd.to_datetime(monthly['Year Month'])

returned_ids = set(returns['Order ID'].dropna()) if 'Order ID' in returns.columns else set()
orders['Returned'] = orders['Order ID'].isin(returned_ids)
return_rate = orders['Returned'].mean() * 100

print('Return rate:', f'{return_rate:.2f}%')
display(discount)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=discount, x='Discount Level', y='Profit Margin', ax=axes[0])
axes[0].set_title('Profit Margin by Discount Level')
sns.lineplot(data=monthly.sort_values('Date'), x='Date', y='Sales', ax=axes[1])
axes[1].set_title('Monthly Sales Trend')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

worst_products = orders.groupby(['Product ID','Product Name'], as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum')).sort_values('Profit').head(10)
display(worst_products)


In [ ]:
# Business recommendations generated from the analysis
best_category = category.loc[category['Profit'].idxmax(), 'Category']
weak_category = category.loc[category['Profit'].idxmin(), 'Category']
best_region = region.loc[region['Profit'].idxmax(), 'Region']
weak_region = region.loc[region['Profit'].idxmin(), 'Region']
worst_product = worst_products.iloc[0]['Product Name']

print('BUSINESS RECOMMENDATIONS')
print(f'1. Protect and scale the {best_category} category because it contributes the strongest total profit.')
print(f'2. Review pricing, discounting and product mix in {weak_category}, the weakest category by total profit.')
print(f'3. Use {best_region} as a benchmark for regional performance and investigate the gap with {weak_region}.')
print('4. Review high-discount transactions where profit margins deteriorate or become negative.')
print(f'5. Investigate the loss-making product {worst_product} for pricing, cost or assortment changes.')
print('6. Use monthly demand patterns for inventory and promotional planning.')

# Portfolio summary
print('\nWorkflow: Raw Excel -> Cleaning -> Feature Engineering -> EDA -> Profitability -> Discounts -> Returns -> Time Series -> Product Analysis -> Recommendations')
